# Model Selection for LMMs and GLMMs in R

## Overview

Model selection for mixed effects models involves two related but distinct decisions:

1. **Random effects structure selection** — how complex should the random effects be? (random intercepts only vs. random intercepts + slopes)
2. **Fixed effects selection** — which predictors should be in the model?

These two decisions use different tools and should be approached in sequence: determine the maximal defensible random effects structure first, then select among fixed effects.

| Tool | Use Case |
|---|---|
| Likelihood ratio test (LRT) | Compare nested models — formally tests whether a more complex model fits significantly better |
| AIC / BIC | Compare non-nested models or rank a set of candidate models |
| REML vs. ML | REML for comparing random effects structures; ML for comparing fixed effects |
| `MuMIn::dredge()` | Automated multi-model comparison across fixed effects combinations |

> **The keep-it-maximal principle (Barr et al. 2013):** In confirmatory analysis, fit the maximal random effects structure justified by the design. Simplify only if the model fails to converge or produces singular fits.

---

## Setup

In [1]:
library(tidyverse)
library(lme4)
library(lmerTest)
library(glmmTMB)
library(MuMIn)        # dredge() for automated model comparison
library(broom.mixed)
library(performance)

set.seed(42)
data(sleepstudy, package = "lme4")

Warning message:
"package 'tidyverse' was built under R version 4.4.3"
Warning message:
"package 'ggplot2' was built under R version 4.4.3"
Warning message:
"package 'purrr' was built under R version 4.4.3"
Warning message:
"package 'dplyr' was built under R version 4.4.3"
Warning message:
"package 'stringr' was built under R version 4.4.3"
── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.0     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.6.0
✔ ggplot2   4.0.2     ✔ tibble    3.2.1
✔ lubridate 1.9.3     ✔ tidyr     1.3.1
✔ purrr     1.2.1     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Warning message:
"package 'lme4' was built under R version 4.4.3"
Loading required package: Matrix


Attaching package: 'Matrix'


Th

---

## REML vs. ML: When to Use Each

This is one of the most common sources of confusion in mixed model fitting.

| Comparison | Estimation Method | Why |
|---|---|---|
| Random effects structures (same fixed effects) | REML | REML gives unbiased variance estimates; LRT on REML likelihoods is valid for nested random effects |
| Fixed effects structures (same random effects) | ML | REML likelihood depends on the fixed effects design matrix; cannot compare different fixed effects with REML |
| Final reported model | REML | Variance component estimates are reported from the REML fit |

In [2]:
# ── Demonstrate REML vs. ML ───────────────────────────────────────────────────
# These two models have DIFFERENT random effects → compare with REML
m_ri   <- lmer(Reaction ~ Days + (1 | Subject),
                data = sleepstudy, REML = TRUE)
m_rs   <- lmer(Reaction ~ Days + (Days | Subject),
                data = sleepstudy, REML = TRUE)

# ── These two models have DIFFERENT fixed effects → refit with ML ─────────────
m_days_ml  <- lmer(Reaction ~ Days + (Days | Subject),
                    data = sleepstudy, REML = FALSE)
m_null_ml  <- lmer(Reaction ~ 1    + (Days | Subject),
                    data = sleepstudy, REML = FALSE)

---

## Step 1: Select Random Effects Structure

Compare nested random effects models using REML likelihood ratio tests. Start with the maximal structure supported by the design and simplify if needed.

In [3]:
# ── Candidate random effects structures ──────────────────────────────────────
m1 <- lmer(Reaction ~ Days + (1 | Subject),
            data = sleepstudy, REML = TRUE)               # random intercepts only
m2 <- lmer(Reaction ~ Days + (Days || Subject),
            data = sleepstudy, REML = TRUE)               # uncorrelated slopes
m3 <- lmer(Reaction ~ Days + (Days | Subject),
            data = sleepstudy, REML = TRUE)               # correlated slopes (maximal)

# ── LRT: does adding random slopes improve fit? ───────────────────────────────
anova(m1, m2, refit = FALSE)   # refit = FALSE keeps REML likelihoods
# Significant chi-square = random slopes improve fit significantly

# ── LRT: does adding intercept-slope correlation improve fit? ─────────────────
anova(m2, m3, refit = FALSE)

# ── Check for singular fit ────────────────────────────────────────────────────
isSingular(m1); isSingular(m2); isSingular(m3)
# TRUE = random effects structure too complex for this data

# ── Select based on LRT results + singularity check ──────────────────────────
# If m3 fits significantly better than m2 and is not singular: use m3
# If singular: fall back to m2 or m1

,npar,AIC,BIC,logLik,-2*log(L),Chisq,Df,Pr(>Chisq)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
m1,4,1794.465,1807.237,-893.2325,1786.465,NA,NA,NA
m2,5,1753.669,1769.634,-871.8346,1743.669,42.79579,1,6.076273e-11


,npar,AIC,BIC,logLik,-2*log(L),Chisq,Df,Pr(>Chisq)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
m2,5,1753.669,1769.634,-871.8346,1743.669,NA,NA,NA
m3,6,1755.628,1774.786,-871.8141,1743.628,0.04102162,1,0.8394962


[1] FALSE

[1] FALSE

[1] FALSE

---

## Step 2: Select Fixed Effects Structure

With random effects determined, compare fixed effects models using ML (not REML) likelihood ratio tests or information criteria.

In [5]:
# ── Simulate a richer dataset with multiple predictors ────────────────────────
n_subj <- 30; n_obs <- 8
subj_re <- rnorm(n_subj, 0, 20)

rich_data <- tibble(
  subject  = rep(paste0("s", 1:n_subj), each = n_obs),
  days     = rep(0:(n_obs-1), times = n_subj),
  caffeine = rnorm(n_subj * n_obs, 200, 50),
  age      = rep(sample(20:60, n_subj, replace = TRUE), each = n_obs),
  reaction = 250 + 10*rep(0:(n_obs-1), times = n_subj) -
             0.05*caffeine + 0.5*age +
             rep(subj_re, each = n_obs) + rnorm(n_subj * n_obs, 0, 25)
)

# ── Fit candidate fixed effects models (ML for comparison) ────────────────────
rf <- list(
  null     = lmer(reaction ~ 1 + (days | subject), data = rich_data, REML = FALSE),
  days     = lmer(reaction ~ days + (days | subject), data = rich_data, REML = FALSE),
  days_caf = lmer(reaction ~ days + caffeine + (days | subject), data = rich_data, REML = FALSE),
  full     = lmer(reaction ~ days + caffeine + age + (days | subject), data = rich_data, REML = FALSE)
)

# ── LRT: sequential model comparison ─────────────────────────────────────────
do.call(anova, c(rf$null, rf$days, rf$days_caf, rf$full))

# ── AIC / BIC comparison table ────────────────────────────────────────────────
map_dfr(names(rf), ~ {
  g <- broom.mixed::glance(rf[[.x]])
  tibble(model = .x, AIC = g$AIC, BIC = g$BIC, logLik = g$logLik)
}) %>%
  mutate(delta_AIC = AIC - min(AIC),
         delta_BIC = BIC - min(BIC)) %>%
  arrange(AIC)

ERROR: Error in anova.lmerModLmerTest(null = new("lmerModLmerTest", vcov_varpar = structure(c(0.146513820764557, : argument "object" is missing, with no default


---

## Multi-Model Inference with `MuMIn::dredge()`

For exploratory analyses, `dredge()` fits all possible subsets of a global model and ranks them by AIC. Use with caution — it is a data-exploration tool, not a substitute for hypothesis-driven analysis.

In [10]:
# ── Fit global model with ML ──────────────────────────────────────────────────
global_model <- lmer(
  reaction ~ days + caffeine + age + (days | subject),
  data    = rich_data,
  REML    = FALSE,
  na.action = na.fail   # required by dredge()
)

# ── Dredge: all fixed effects subsets ─────────────────────────────────────────
dredge_results <- MuMIn::dredge(global_model)
print(dredge_results)
# delta < 2: models with substantial support
# delta 2-7: considerably less support
# delta > 10: essentially no support

# ── Model averaging: average coefficients weighted by AIC weights ─────────────
# Use for inference when multiple models have similar support
avg_model <- MuMIn::model.avg(dredge_results, subset = delta < 4)
summary(avg_model)
# 'full' coefficients: averaged over all models (zeroed for absent terms)
# 'conditional' coefficients: averaged only over models containing that term

# ── Variable importance ───────────────────────────────────────────────────────
MuMIn::sw(avg_model)
# Sum of AIC weights across all models containing each term
# Higher = predictor appears in more high-weight models

Fixed term is "(Intercept)"

Warning message in checkConv(attr(opt, "derivs"), opt$par, ctrl = control$checkConv, :
"Model failed to converge with max|grad| = 0.00308677 (tol = 0.002, component 1)
  See ?lme4::convergence and ?lme4::troubleshooting."


Global model call: lmer(formula = reaction ~ days + caffeine + age + (days | subject), 
    data = rich_data, REML = FALSE, na.action = na.fail)
---
Model selection table 
  (Intrc)     age    caffn  days df    logLik   AICc delta weight
7   276.4         -0.05488 9.626  7 -1128.240 2271.0  0.00  0.438
5   265.4                  9.582  6 -1129.685 2271.7  0.77  0.298
8   272.5 0.10650 -0.05558 9.627  8 -1128.187 2273.0  2.03  0.158
6   262.7 0.07044          9.582  7 -1129.662 2273.8  2.84  0.106
1   310.3                         5 -1154.676 2319.6 48.65  0.000
3   320.3         -0.04439        6 -1153.790 2319.9 48.98  0.000
2   307.5 0.07045                 6 -1154.653 2321.7 50.70  0.000
4   316.5 0.09953 -0.04508        7 -1153.744 2322.0 51.01  0.000
Models ranked by AICc(x) 
Random terms (all models): 
  days | subject




Call:
model.avg(object = dredge_results, subset = delta < 4)

Component model call: 
lmer(formula = reaction ~ <4 unique rhs>, data = rich_data, REML = 
     FALSE, na.action = na.fail)

Component models: 
    df   logLik    AICc delta weight
23   7 -1128.24 2270.96  0.00   0.44
3    6 -1129.68 2271.73  0.77   0.30
123  8 -1128.19 2273.00  2.03   0.16
13   7 -1129.66 2273.81  2.84   0.11

Term codes: 
     age caffeine     days 
       1        2        3 

Model-averaged coefficients:  
(full average) 
             Estimate Std. Error Adjusted SE z value Pr(>|z|)    
(Intercept) 271.05655   11.21753    11.26214  24.068   <2e-16 ***
caffeine     -0.03282    0.03663     0.03671   0.894    0.371    
days          9.60830    0.83491     0.83926  11.448   <2e-16 ***
age           0.02429    0.17317     0.17403   0.140    0.889    
 
(conditional average) 
             Estimate Std. Error Adjusted SE z value Pr(>|z|)    
(Intercept) 271.05655   11.21753    11.26214  24.068   <2e-16 ***
caf

days caffeine age 
Sum of weights:      1.00 0.60     0.26
N containing models:    4    2        2

---

## Model Selection for GLMMs

The same principles apply to GLMMs. REML is not available for GLMs, so all comparisons use ML (which is the default for `glmer` and `glmmTMB`).

In [11]:
# ── Simulate binary outcome data ──────────────────────────────────────────────
n_sites <- 25; n_v <- 4; N <- n_sites * n_v
site_re <- rnorm(n_sites, 0, 0.8)

glmm_data <- tibble(
  site   = rep(paste0("s", 1:n_sites), each = n_v),
  temp   = rnorm(N, 15, 3),
  precip = rnorm(N, 50, 20),
  lo     = -1 + 0.12*temp + 0.005*precip + rep(site_re, each = n_v),
  y      = rbinom(N, 1, plogis(lo))
) %>% select(-lo)

# ── Candidate GLMM models ─────────────────────────────────────────────────────
gm1 <- lme4::glmer(y ~ 1      + (1|site), data = glmm_data, family = binomial)
gm2 <- lme4::glmer(y ~ temp   + (1|site), data = glmm_data, family = binomial)
gm3 <- lme4::glmer(y ~ precip + (1|site), data = glmm_data, family = binomial)
gm4 <- lme4::glmer(y ~ temp + precip + (1|site), data = glmm_data, family = binomial)

# ── AIC comparison ────────────────────────────────────────────────────────────
AIC(gm1, gm2, gm3, gm4) %>%
  mutate(delta_AIC = AIC - min(AIC)) %>%
  arrange(AIC)

# ── LRT for nested models ─────────────────────────────────────────────────────
anova(gm1, gm2)   # does adding temp improve fit?
anova(gm2, gm4)   # does adding precip to temp model improve fit?

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')

boundary (singular) fit: see help('isSingular')



,df,AIC,delta_AIC
,<dbl>,<dbl>,<dbl>
gm2,3,99.94302,0.000000
gm1,2,101.24459,1.301569
gm4,4,101.94002,1.996997
gm3,3,103.23010,3.287079


,npar,AIC,BIC,logLik,-2*log(L),Chisq,Df,Pr(>Chisq)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
gm1,2,101.24459,106.4549,-48.62230,97.24459,NA,NA,NA
gm2,3,99.94302,107.7585,-46.97151,93.94302,3.301569,1,0.06921373


,npar,AIC,BIC,logLik,-2*log(L),Chisq,Df,Pr(>Chisq)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
gm2,3,99.94302,107.7585,-46.97151,93.94302,NA,NA,NA
gm4,4,101.94002,112.3607,-46.97001,93.94002,0.003002965,1,0.9562983


---

## Common Pitfalls

**1. Comparing models with different fixed effects using REML**  
REML likelihoods are not comparable across different fixed effects designs. Always refit with `REML = FALSE` before comparing fixed effects models with LRT or AIC.

**2. Simplifying random effects before fixed effects are determined**  
Determine random effects structure first using REML, then fix random effects and select fixed effects with ML. Doing this in the wrong order can lead to incorrect variance estimates.

**3. Using dredge() for confirmatory hypothesis testing**  
`dredge()` is an exploratory tool. Running it on confirmatory data and reporting the best model as if it were hypothesis-driven is p-hacking. Pre-register your model or treat dredge results as hypothesis-generating.

**4. Treating ΔAIC < 2 as definitive support**  
ΔAIC < 2 indicates comparable support — it does not mean models are equivalent. Report and discuss competing models rather than selecting a single winner.

**5. Ignoring singular fit when building random effects**  
A singular fit during random effects selection means the data cannot support that complexity. This is useful diagnostic information — simplify the random effects and note it in the methods.

**6. Not reporting model selection criteria in methods**  
Always document which information criterion was used, whether REML or ML was used for each comparison, and what the candidate model set was. Reproducibility requires this.

---
*r_methods_library · Samantha McGarrigle · [github.com/samantha-mcgarrigle](https://github.com/samantha-mcgarrigle)*